In [18]:
import random
import os
import pandas as pd

from sklearn.model_selection import KFold, StratifiedKFold

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW, BertModel, BertConfig, AutoConfig
import os
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score,f1_score, classification_report,recall_score, precision_score
import numpy as np
from sklearn.model_selection import train_test_split


In [19]:
#CONFIG

#General 
CATEG = "pc" #Category name in dataset
MODEL_OUT_DIR = 'model/' #Path to save model and files


TRAINING_DATA = 'train_sub_sub_stratified v2.csv' #Path Training data
#TEST_DATA  = 'test_total_stratified.csv' #Path Test Data

#Files names
NOME_AMOSTRA = "MODELO 09 - BERT BASE - AMOSTRA SUB"
NOME_AMOSTRA_ARQ = f"MODELO09_{CATEG}_"



## Model Configurations
MAX_LEN = 122
BATCH_SIZE = 32 #Batch size
LR = 5e-5 #Learning rate
NUM_EPOCHS = 5
NUM_THREADS = 1  ## Number of threads for collecting dataset
MODEL_NAME = 'neuralmind/bert-base-portuguese-cased' #Model name


# Total Folds
NUM_FOLDS = 3
NUM_CORES = os.cpu_count()

#Device 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")



In [20]:
treino_df = pd.read_csv (TRAINING_DATA) 

dados_treinamento = treino_df.loc[:,["text",CATEG]]
dados_treinamento.head()

,text,pc
0,"obrigado por esse gif tão bom, @gangdaflorzinh...",0
1,"""o brasil nunca deixou de ser um lugar vivo cu...",0
2,o brasileiro sofreu muito com a inflação nos a...,0
3,@heliotelho @deltanmd eu votei não!\n\nfui ele...,0
4,obrigado por tudo! amo você<u+2764><u+fe0f>,0


In [21]:
#Categoria 



seed_val = 22
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME,max_lenght=MAX_LEN)



In [22]:
class GerarDataset(Dataset):
  def __init__(self, data, tokenizer):
        self.df = data.reset_index()
        self.tokenizer = tokenizer

  def __len__(self):
        return self.df.shape[0]

  def __getitem__(self, index):
        #Select the sentence and label at the specified index in the data frame
        texto_tratado = self.df.loc[index, 'text']
        if pd.isna(texto_tratado):
          texto_tratado = ""    
        target = self.df.loc[index, CATEG]
        #identifier = self.df.loc[index, 'id']
        tokens = tokenizer(texto_tratado,
                           padding='max_length',
                           max_length=MAX_LEN,
                           add_special_tokens=True,
                           return_tensors='pt',
                           truncation=True)
        
        input_ids = tokens["input_ids"].clone().detach()
        #torch.tensor(tokens["input_ids"]) codigo antigo - warning 
        attention_mask = tokens["attention_mask"].clone().detach()
        target = torch.tensor(target, dtype=torch.long)
        #torch.tensor(target, dtype=torch.long)
        return input_ids, attention_mask, target

def MontarModelo():
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, classifier_dropout  =0.2).to(DEVICE)
    model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR,   eps = 1e-8)
    criterion = nn.CrossEntropyLoss()
    return model, optimizer, criterion


In [23]:
def TreinarModelo(model, criterion, optimizer, train_loader, val_loader, epochs):
    #Medidas agregadas do fold
    print("Iniciando o treinamento do modelo...")
    train_losses = []
    val_losses = []
    vals_f1 = []
    
    precisao_validacao = []
    recall_validacao = []
    acc_validacao = []
   
    epoca_ref = []
    max_f1 = 0  
    folds = []
    nomes = []
    features = []

    f1s_class1 = []
    precisions_class1 = []
    recalls_class1 = []

    model.to(DEVICE)

    #Para cada época
    for epoch in range(epochs):
        model.train() #Acerta o modelo para treinamento
        train_loss = 0 #zera a loss
        target_train = []
        preditos_train = []

        for i, (input_ids, attention_mask, target) in enumerate(iterable=train_loader):
            optimizer.zero_grad()
            input_ids, attention_mask, target = input_ids.to(DEVICE), attention_mask.to(DEVICE), target.to(DEVICE)
            # Ajustando as formas dos tensores para [batch_size, sequence_length]
            input_ids = input_ids.squeeze(1)  # Remove a dimensão extra
            attention_mask = attention_mask.squeeze(1)  # Remove a dimensão extra
            #print(f"input_ids shape: {input_ids.shape}, attention_mask shape: {attention_mask.shape}")

            output = model(input_ids=input_ids, attention_mask=attention_mask)

            preditos_train.extend(torch.argmax(output.logits, 1).to("cpu").tolist())

            loss = criterion(output.logits,target)
            loss.backward()
            optimizer.step() 
            train_loss += loss.item()
            target_train.extend(target.to("cpu").tolist())
        
        target_train = np.array(target_train)
        preditos_train = np.array(preditos_train)



        f1_treinamento = f1_score(target_train, preditos_train, average='binary')

        print("\n***")
        epoca_padrao  = epoch + 1

        #VALIDACAO
        # Validação no final de cada época
        f1, acc, precision, recall, val_loss_medio, f1_class1, precision_class1, recall_class1 = CalcularValidacao(model=model, criterion=criterion, val_loader=val_loader)

        print(f"Epoch {epoch + 1}, Training Loss: {train_loss / len(train_loader)}")
        print(f"Epoch {epoch + 1}, Validation Loss: {val_loss_medio}")
        #print(f"Epoch {epoch + 1}, Reference F1: {max_f1}, F1: {f1}")
        #print(f"Epoch {epoch + 1}, Training F1 (Class 1): {f1_treinamento}, F1 Classe 1: {f1_class1}")
        #print(f"Epoch {epoch + 1}, Recall (Class 1): {recall_class1}, Precision (Class 1): {precision_class1}")
        #print((f1 > max_f1 and f1_class1 > 0.5))
        print("\n***")

        # Registrar valores de cada época
        
        nomes.append(NOME_AMOSTRA)
        features.append(CATEG)
        epoca_ref.append(epoch + 1)
        train_losses.append(train_loss / len(train_loader))
        val_losses.append(val_loss_medio)
        vals_f1.append(f1)
        f1s_class1.append(f1_class1)
        #f1_nonpop.append(f1_per_class[0])
        
        acc_validacao.append(acc)
        precisao_validacao.append(precision)
        precisions_class1.append(precision_class1)
        recall_validacao.append(recall)
        recalls_class1.append(recall_class1)
        folds.append(fold)

        # Salvar o melhor modelo baseado no F1
        if  (f1 > max_f1 and f1_class1 >= 0.55  and precision_class1 >= 0.55 and recall_class1 >= 0.55):
            print("\nSalvando o modelo com F1 superior...")
            print("\n****")
            modelo_nome = f"{NOME_AMOSTRA_ARQ}_fold_{fold}_epoch_{epoch + 1}.bin"
            modelo_caminho = os.path.join(MODEL_OUT_DIR, modelo_nome)
            torch.save(model.state_dict(), modelo_caminho)

            max_f1 = f1 

    
    resultados = pd.DataFrame({
        "Model name": nomes,
        "Feature": features,
        "Epoch": epoca_ref,
        "Train Loss": train_losses,
        "Validation Loss": val_losses,
        "Validation F1": vals_f1,
        "Validation Accuracy": acc_validacao,
        "Validation Precision": precisao_validacao,
        "Validation Recall": recall_validacao,
        "Validation F1 (Class 1)": f1s_class1,
        "Validation Precision(Class 1)": precisions_class1,
        "Validation Recall(Class 1)": recalls_class1,
        "Fold": folds
    })

    # Salvar o DataFrame em CSV
    resultados_csv = os.path.join(MODEL_OUT_DIR, f"Result_{NOME_AMOSTRA_ARQ}{CATEG}_fold_{fold}.csv")
    resultados.to_csv(resultados_csv, index=False)
    print(f"Resultados salvos em {resultados_csv}")

def CalcularValidacao(model, criterion, val_loader):
    model.eval()  # Coloca o modelo em modo de avaliação
    y_real, y_pred = [], []  # Armazenam as previsões e os rótulos reais
    val_loss_total = 0  # Acumula a perda de validação
    with torch.no_grad():
        for i, (input_ids, attention_mask, target) in enumerate(iterable=val_loader):
            input_ids, attention_mask, target = input_ids.to(DEVICE), attention_mask.to(DEVICE), target.to(DEVICE)
            labels = []
            #print(f"input_ids shape: {input_ids.shape}, attention_mask shape: {attention_mask.shape}")
            # Classificação
            input_ids = input_ids.squeeze(1)  # Remove a dimensão extra
            attention_mask = attention_mask.squeeze(1)  # Remove a dimensão extra
            
            output = model(input_ids=input_ids, attention_mask=attention_mask)
            val_loss = criterion(output.logits,target)
            val_loss_total += val_loss.item()  # Soma a perda para calcular a média depois
            
            # Previsões
            preditos = torch.argmax(output.logits, 1).to("cpu").tolist()
            y_pred.extend(preditos)  # Armazena as previsões
        
            y_real.extend(target.to("cpu").tolist()) # Armazena os rótulos reais
    # Converte as listas para numpy arrays para calcular as métricas
    y_real = np.array(y_real)
    y_pred = np.array(y_pred)
    # Calcula as principais métricas de classificação
    f1 = f1_score(y_real, y_pred, average='macro',zero_division=0)
    acc = accuracy_score(y_real, y_pred)
    precision = precision_score(y_real, y_pred, average='macro',zero_division=0)
    recall = recall_score(y_real, y_pred, average='macro',zero_division=0)
    
    f1_class1 = f1_score(y_real, y_pred, average='binary',zero_division=0)
    precision_class1 = precision_score(y_real, y_pred, average='binary',zero_division=0)
    recall_class1 = recall_score(y_real, y_pred, average='binary',zero_division=0)
    
    # Exibe um relatório de classificação detalhado
    print("*** Validação ***")
    print(classification_report(y_real, y_pred))
    # Retorna F1, acurácia, precisão, recall e a perda de validação média
    val_loss_medio = val_loss_total / len(val_loader)  # Calcula a perda média
    return f1, acc, precision, recall, val_loss_medio, f1_class1, precision_class1, recall_class1

In [24]:
# Define k-fold cross-validation
k_folds = NUM_FOLDS
skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

# Initialize lists to store accuracies for each fold
fold_F1 = []

#ref - https://vtiya.medium.com/lets-code-k-fold-validation-on-bert-722f9438f932
for fold, (train_indices, val_indices) in enumerate(skf.split(dados_treinamento['text'], dados_treinamento[CATEG])):
  print(f"Training Fold {fold+1}/{k_folds}")

  # Split dataset into train and validation sets for the current fold
  train_dataset =  dados_treinamento.iloc[train_indices,:]
  val_dataset = dados_treinamento.iloc[val_indices,:]
  train_res = train_dataset.reset_index(drop = True)
  val_res = val_dataset.reset_index(drop = True)

  # Tokenizando e transformando em inputs (att masks, labels, etc.)
  train_data = GerarDataset(train_res,tokenizer)
  val_data = GerarDataset(val_res,tokenizer)

  # Criando data loaders
  train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
  val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)

  del(train_dataset,val_dataset,train_res,val_res)

  model, optimizer, criterion = MontarModelo()

  TreinarModelo(model, criterion, optimizer, train_loader, val_loader, NUM_EPOCHS)





Training Fold 1/3


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Iniciando o treinamento do modelo...

***
*** Validação ***
              precision    recall  f1-score   support

           0       0.77      1.00      0.87       232
           1       0.00      0.00      0.00        68

    accuracy                           0.77       300
   macro avg       0.39      0.50      0.44       300
weighted avg       0.60      0.77      0.67       300

Epoch 1, Training Loss: 0.5554506057187131
Epoch 1, Validation Loss: 0.5169148504734039

***


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



***
*** Validação ***
              precision    recall  f1-score   support

           0       0.94      0.91      0.92       232
           1       0.71      0.81      0.76        68

    accuracy                           0.88       300
   macro avg       0.83      0.86      0.84       300
weighted avg       0.89      0.88      0.89       300

Epoch 2, Training Loss: 0.38030965783094106
Epoch 2, Validation Loss: 0.3066122516989708

***

Salvando o modelo com F1 superior...

****

***
*** Validação ***
              precision    recall  f1-score   support

           0       0.83      0.97      0.90       232
           1       0.79      0.32      0.46        68

    accuracy                           0.83       300
   macro avg       0.81      0.65      0.68       300
weighted avg       0.82      0.83      0.80       300

Epoch 3, Training Loss: 0.20864351838827133
Epoch 3, Validation Loss: 0.5070992400869727

***

***
*** Validação ***
              precision    recall  f1-score  

...

Iniciando o treinamento do modelo...

***
*** Validação ***
              precision    recall  f1-score   support

           0       0.82      0.98      0.89       231
           1       0.83      0.28      0.41        69

    accuracy                           0.82       300
   macro avg       0.82      0.63      0.65       300
weighted avg       0.82      0.82      0.78       300

Epoch 1, Training Loss: 0.5121000707149506
Epoch 1, Validation Loss: 0.4148711316287518

***

***
*** Validação ***
              precision    recall  f1-score   support

           0       0.94      0.85      0.89       231
           1       0.62      0.81      0.70        69

    accuracy                           0.84       300
   macro avg       0.78      0.83      0.80       300
weighted avg       0.87      0.84      0.85       300

Epoch 2, Training Loss: 0.3201585835532138
Epoch 2, Validation Loss: 0.31164039522409437

***

Salvando o modelo com F1 superior...

****

***
*** Validação ***
         

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Iniciando o treinamento do modelo...

***
*** Validação ***
              precision    recall  f1-score   support

           0       0.77      1.00      0.87       231
           1       0.00      0.00      0.00        69

    accuracy                           0.77       300
   macro avg       0.39      0.50      0.44       300
weighted avg       0.59      0.77      0.67       300

Epoch 1, Training Loss: 0.5295966095045993
Epoch 1, Validation Loss: 0.47331770807504653

***


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



***
*** Validação ***
              precision    recall  f1-score   support

           0       0.94      0.81      0.87       231
           1       0.57      0.81      0.67        69

    accuracy                           0.81       300
   macro avg       0.75      0.81      0.77       300
weighted avg       0.85      0.81      0.82       300

Epoch 2, Training Loss: 0.3760268005885576
Epoch 2, Validation Loss: 0.39089091271162035

***

Salvando o modelo com F1 superior...

****

***
*** Validação ***
              precision    recall  f1-score   support

           0       0.94      0.80      0.87       231
           1       0.56      0.84      0.67        69

    accuracy                           0.81       300
   macro avg       0.75      0.82      0.77       300
weighted avg       0.86      0.81      0.82       300

Epoch 3, Training Loss: 0.24528552631014272
Epoch 3, Validation Loss: 0.4519243985414505

***

***
*** Validação ***
              precision    recall  f1-score  